In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ipywidgets import interact, Dropdown, IntSlider, Checkbox, RadioButtons
import ipywidgets as widgets
from IPython.display import display, Markdown

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


In [2]:
# Cargar datos
df = pd.read_csv("online_shoppers_intention.csv")

print(f"Dataset cargado correctamente: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()


Dataset cargado correctamente: 12330 filas x 18 columnas


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [3]:
# Tipos de variables
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

print("Variables numéricas:")
print(num_cols)
print("\nVariables categóricas:")
print(cat_cols)


Variables numéricas:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

Variables categóricas:
['Month', 'VisitorType', 'Weekend', 'Revenue']


## Exploración base del dataset

Esta sección ayuda a documentar el notebook, tal como pide PB-06, y deja evidencia de comprensión del conjunto de datos.


In [4]:
# Resumen general
display(df.describe(include="all").T.fillna(""))


/tmp/ipykernel_3455/1118453263.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  display(df.describe(include="all").T.fillna(""))


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Administrative,12330.0,,,,2.315166,3.321784,0.0,0.0,1.0,4.0,27.0
Administrative_Duration,12330.0,,,,80.818611,176.779107,0.0,0.0,7.5,93.25625,3398.75
Informational,12330.0,,,,0.503569,1.270156,0.0,0.0,0.0,0.0,24.0
Informational_Duration,12330.0,,,,34.472398,140.749294,0.0,0.0,0.0,0.0,2549.375
ProductRelated,12330.0,,,,31.731468,44.475503,0.0,7.0,18.0,38.0,705.0
ProductRelated_Duration,12330.0,,,,1194.74622,1913.669288,0.0,184.1375,598.936905,1464.157214,63973.52223
BounceRates,12330.0,,,,0.022191,0.048488,0.0,0.0,0.003112,0.016813,0.2
ExitRates,12330.0,,,,0.043073,0.048597,0.0,0.014286,0.025156,0.05,0.2
PageValues,12330.0,,,,5.889258,18.568437,0.0,0.0,0.0,0.0,361.763742
SpecialDay,12330.0,,,,0.061427,0.198917,0.0,0.0,0.0,0.0,1.0


In [5]:
def explorar(variable, bins, mostrar_boxplot, tipo_grafico, filtro_revenue):
    data = df.copy()

    # Filtro opcional por target
    if filtro_revenue != "Todos":
        valor = True if filtro_revenue == "True" else False
        data = data[data["Revenue"] == valor]

    if data.empty:
        print("No hay datos para el filtro seleccionado.")
        return

    es_numerica = variable in num_cols
    es_categorica = variable in cat_cols

    plt.figure(figsize=(10, 5))

    if tipo_grafico == "Histograma":
        if es_numerica:
            sns.histplot(data[variable].dropna(), bins=bins, kde=True)
            plt.title(f"Histograma de {variable}")
            plt.xlabel(variable)
            plt.ylabel("Frecuencia")
        else:
            print(f"La variable '{variable}' no es numérica. Usa 'Conteo' para variables categóricas.")
            plt.close()

    elif tipo_grafico == "Boxplot":
        if es_numerica:
            sns.boxplot(x=data[variable])
            plt.title(f"Boxplot de {variable}")
            plt.xlabel(variable)
        else:
            print(f"La variable '{variable}' no es numérica. El boxplot aplica a variables numéricas.")
            plt.close()

    elif tipo_grafico == "Conteo":
        if es_categorica:
            order = data[variable].astype(str).value_counts().index
            sns.countplot(x=data[variable].astype(str), order=order)
            plt.xticks(rotation=45, ha="right")
            plt.title(f"Conteo de {variable}")
            plt.xlabel(variable)
            plt.ylabel("Frecuencia")
        else:
            print(f"La variable '{variable}' es numérica. Usa histograma o boxplot.")
            plt.close()
    else:
        print("Tipo de gráfico no reconocido.")
        plt.close()

    if plt.get_fignums():
        plt.tight_layout()
        plt.show()

    # Boxplot adicional opcional
    if mostrar_boxplot and es_numerica and tipo_grafico != "Boxplot":
        plt.figure(figsize=(8, 2.8))
        sns.boxplot(x=data[variable])
        plt.title(f"Boxplot adicional de {variable}")
        plt.xlabel(variable)
        plt.tight_layout()
        plt.show()

    # Resumen textual
    print("\nResumen de la variable:")
    if es_numerica:
        display(data[variable].describe().to_frame(name=variable))
    elif es_categorica:
        display(data[variable].astype(str).value_counts().to_frame(name="frecuencia"))


## Interfaz interactiva

Este bloque constituye el **prototipo PB-06**: un stakeholder no técnico puede explorar los datos sin escribir código.


In [6]:
interact(
    explorar,
    variable=Dropdown(
        options=df.columns.tolist(),
        value=df.columns.tolist()[0],
        description="Variable:"
    ),
    bins=IntSlider(
        min=5, max=100, step=5, value=30,
        description="Bins:"
    ),
    mostrar_boxplot=Checkbox(
        value=False,
        description="Boxplot extra"
    ),
    tipo_grafico=RadioButtons(
        options=["Histograma", "Boxplot", "Conteo"],
        value="Histograma",
        description="Gráfico:"
    ),
    filtro_revenue=RadioButtons(
        options=["Todos", "True", "False"],
        value="Todos",
        description="Revenue:"
    )
);


interactive(children=(Dropdown(description='Variable:', options=('Administrative', 'Administrative_Duration', …